# Merge Agent Notebook (Sponsor + Speaker/Artist)

This notebook uses the same patterns as the sponsor and speaker agents in a simpler combined flow.
It takes user event input once, runs both agents, shows markdown tables, and saves outputs to files.

## Cell 1 - Install Dependencies

In [ ]:
%pip install langchain langchain-community langchain-core langchain-openai chromadb sentence-transformers pandas python-dotenv tavily-python -q

## Cell 2 - Imports and Setup

In [1]:
import os
import json
from typing import List, Dict, Any
from dotenv import load_dotenv
from IPython.display import display, Markdown

from tavily import TavilyClient
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

SPONSOR_DB_PATH = "./sponsor_vector_db"
SPEAKER_DB_PATH = "./speaker_vector_db"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "openai/gpt-oss-120b"

embedding_fn = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
sponsor_db = Chroma(persist_directory=SPONSOR_DB_PATH, embedding_function=embedding_fn)
speaker_db = Chroma(persist_directory=SPEAKER_DB_PATH, embedding_function=embedding_fn)

llm = ChatOpenAI(
    model=LLM_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.1,
    max_tokens=3000
)

tavily_client = TavilyClient(api_key=TAVILY_API_KEY) if TAVILY_API_KEY else None

print("Setup complete")
print(f"OpenRouter key: {'Yes' if OPENROUTER_API_KEY else 'No'}")
print(f"Tavily key: {'Yes' if TAVILY_API_KEY else 'No'}")
print(f"Sponsor docs: {sponsor_db._collection.count()}")
print(f"Speaker docs: {speaker_db._collection.count()}")

c:\Users\KISHORE S\anaconda3\envs\agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\KISHORE S\AppData\Local\Temp\ipykernel_39416\196649473.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_fn = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5127.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_id

Setup complete
OpenRouter key: Yes
Tavily key: Yes
Sponsor docs: 200
Speaker docs: 200


## Cell 3 - Shared Helper Functions

In [2]:
def call_llm(prompt: str) -> str:
    response = llm.invoke([HumanMessage(content=prompt)])
    return str(response.content).strip()

def tavily_search(query: str, max_results: int = 5) -> List[Dict[str, str]]:
    if tavily_client is None:
        return [{"title": "search_error", "url": "", "snippet": "Missing TAVILY_API_KEY"}]

    try:
        res = tavily_client.search(
            query=query,
            max_results=max_results,
            include_answer=False,
            include_raw_content=False,
            search_depth="advanced"
        )
        out = []
        for r in res.get("results", []):
            out.append({
                "title": r.get("title", ""),
                "url": r.get("url", ""),
                "snippet": r.get("content", "")
            })
        return out
    except Exception as e:
        return [{"title": "search_error", "url": "", "snippet": str(e)}]

## Cell 4 - Sponsor Agent Function

In [3]:
def run_sponsor_agent(user_event: Dict[str, Any]) -> str:
    retrieval_query = (
        f"{user_event['category']} conference sponsors in {user_event['location']} "
        f"audience size {user_event.get('audience_size', 'large')} budget {user_event.get('budget', 'medium')} "
        "recent sponsorship and marketing spend in last 12 months"
    )

    docs = sponsor_db.similarity_search(retrieval_query, k=15)
    context_docs = [d.page_content for d in docs]
    context = "\n\n".join(context_docs)

    sponsor_names = []
    seen = set()
    for d in context_docs:
        name = d.split(" is a ")[0].strip() if " is a " in d else d[:60].strip()
        if name and name not in seen:
            seen.add(name)
            sponsor_names.append(name)

    web_profiles = []
    for name in sponsor_names[:8]:
        q = f"{name} sponsorship {user_event['category']} {user_event['location']} last 12 months marketing spend"
        hits = tavily_search(q, max_results=5)
        web_profiles.append({"sponsor_name": name, "query": q, "hits": hits})

    prompt = (
        "You are a sponsorship strategist.\n\n"
        f"Event input: {json.dumps(user_event, ensure_ascii=True)}\n\n"
        "Internal sponsor context:\n"
        f"{context}\n\n"
        "Tavily web evidence:\n"
        f"{json.dumps(web_profiles, ensure_ascii=True, indent=2)}\n\n"
        "Task: recommend top 5 sponsors and create custom proposal lines.\n"
        "Prioritize by industry relevance, geography, and sponsorship frequency.\n\n"
        "Return strict markdown with two tables:\n"
        "1) Sponsor Prioritization Table with columns:\n"
        "| Rank | Sponsor | Industry | Geography Fit | Sponsorship Frequency | Marketing Spend Signal | Priority Score (/100) | Why Selected |\n"
        "2) Custom Sponsorship Proposal Table with columns:\n"
        "| Sponsor | Recommended Tier | Proposal Theme | Key Audience Benefit | Suggested Deliverables | Estimated Ask |"
    )

    return call_llm(prompt)

## Cell 5 - Speaker/Artist Agent Function

In [4]:
def run_speaker_agent(user_event: Dict[str, Any]) -> str:
    retrieval_query = (
        f"{user_event['event_topic']} {user_event['role_type']} candidates for {user_event['location']} "
        f"audience size {user_event.get('audience_size', 'large')} influence and proven speaking"
    )

    docs = speaker_db.similarity_search(retrieval_query, k=18)
    context_docs = [d.page_content for d in docs]
    context = "\n\n".join(context_docs)

    candidate_names = []
    seen = set()
    for d in docs:
        name = str(d.metadata.get("profile_name", "")).strip()
        if name and name not in seen:
            seen.add(name)
            candidate_names.append(name)

    web_profiles = []
    for name in candidate_names[:8]:
        q = f"{name} LinkedIn {user_event['event_topic']} speaker profile followers publications keynote"
        hits = tavily_search(q, max_results=5)
        web_profiles.append({"name": name, "query": q, "hits": hits})

    prompt = (
        "You are an event programming strategist.\n\n"
        f"Event input: {json.dumps(user_event, ensure_ascii=True)}\n\n"
        "Internal speaker context:\n"
        f"{context}\n\n"
        "Tavily web evidence:\n"
        f"{json.dumps(web_profiles, ensure_ascii=True, indent=2)}\n\n"
        "Task: recommend top 5 speaker/artist profiles and agenda mapping.\n"
        "Prioritize topic fit, past speaking/performance, and influence.\n\n"
        "Return strict markdown with two tables:\n"
        "1) Speaker/Artist Prioritization Table with columns:\n"
        "| Rank | Name | Type | Topic Fit | Past Experience Evidence | Influence Signals | Recommended Session | Priority Score (/100) |\n"
        "2) Agenda Mapping Table with columns:\n"
        "| Agenda Slot | Topic | Speaker/Artist | Session Format | Why This Match |"
    )

    return call_llm(prompt)

## Cell 6 - User Input (Single Place)

In [5]:
sponsor_event_input = {
    "category": "AI",
    "location": "India",
    "audience_size": 2500,
    "budget": "60 Lakhs",
    "objective": "Find relevant sponsors with active recent spend"
}

speaker_event_input = {
    "event_topic": "Applied AI in Enterprise",
    "role_type": "speaker",
    "location": "India",
    "audience_size": 2500,
    "budget": "60 Lakhs",
    "goal": "Find high-impact speakers with proven stage presence"
}

## Cell 7 - Run Both Agents

In [6]:
sponsor_result = run_sponsor_agent(sponsor_event_input)
speaker_result = run_speaker_agent(speaker_event_input)

print("Sponsor agent output")
display(Markdown(sponsor_result))

print("Speaker/Artist agent output")
display(Markdown(speaker_result))

Sponsor agent output


**Sponsor Prioritization for “AI India 2026” (Audience ≈ 2 500, Budget ≈ ₹60 Lakh)**  

| Rank | Sponsor | Industry | Geography Fit* | Sponsorship Frequency** | Marketing‑Spend Signal*** | Priority Score (/100) | Why Selected |
|------|---------|----------|----------------|--------------------------|---------------------------|-----------------------|--------------|
| 1 | **Microsoft** | Cloud / AI / Enterprise Software | Very strong – $17.5 B AI‑infrastructure commitment in India, large skilling programmes | Sponsored 5 AI conferences in 2025 (Gold tier) | Massive recent spend (₹1 200 Cr ≈ $17.5 B) plus aggressive brand‑building | **100** | Highest recent financial commitment to AI in India, proven event partnership, and a clear “AI‑first” narrative that matches the conference theme. |
| 2 | **OpenAI** | Generative AI / LLM | Strong – 200‑300 Cr ≈ ₹2‑3 B annual ad spend, free‑tier rollout, New Delhi office | Gold sponsor at AI Conference 13 (Mumbai 2025) | Very active marketing push in India (₹200‑300 Cr) and user‑acquisition campaigns | **90** | Aggressive brand‑building in India, huge buzz around ChatGPT Go, and a perfect audience of developers & product teams. |
| 3 | **Adobe** | Creative AI / Digital Media | Good – AI‑focused fellowship, Firefly adoption high in India, ad spend up 30% YoY | Gold sponsor at 4 AI conferences (Delhi & Mumbai 2025) | Strong spend on AI‑driven marketing tools (₹12 L stipend + $1.4 B global ad spend) | **90** | Wants to showcase Firefly & generative‑design tools to a tech‑savvy crowd; proven event sponsor. |
| 4 | **IBM** | Hybrid Cloud / AI / Quantum | Good – 5 M youth‑skill pledge, AI labs in Bengaluru & Lucknow | Gold sponsor at 2 AI conferences (Mumbai & Delhi 2025) | Ongoing skill‑building spend (₹5 Cr + large R&D budget) | **80** | Positioning as “AI for enterprise transformation” aligns with senior‑level attendees; solid Indian presence. |
| 5 | **Intel** | Semiconductor / Edge AI | Good – Partnerships with Tata, AI‑mission involvement, chip‑fab announcements | Gold sponsor at 5 AI conferences (Delhi & Mumbai 2025) | Moderate spend on partnership programmes (no explicit ad budget) | **80** | Needs ecosystem exposure for its AI‑optimized silicon; can offer hardware demos and developer labs. |

\* **Geography Fit** – 25 pts = global‑scale India investment; 20 pts = significant India‑focused programmes; 15 pts = limited India presence.  
\** **Sponsorship Frequency** – 25 pts = ≥ 4 past AI‑event sponsorships; 20 pts = 2‑3; 15 pts = 1.  
\*** **Marketing‑Spend Signal** – 25 pts = ≥ ₹200 Cr/yr announced spend; 20 pts = ₹50‑199 Cr; 15 pts = ₹10‑49 Cr; 10 pts = < ₹10 Cr or indirect spend.

---

### Custom Sponsorship Proposals  

| Sponsor | Recommended Tier | Proposal Theme | Key Audience Benefit |
|---------|------------------|----------------|----------------------|
| **Microsoft** | **Platinum (₹20 L)** | *“AI‑First India: Building Sovereign Cloud & Skilling”* | Position Microsoft as the backbone of India’s AI ecosystem; direct access to senior engineers, CTOs, and policy‑makers who will influence cloud‑adoption decisions. |
| **OpenAI** | **Gold (₹15 L)** | *“Generative Intelligence Playground”* | Showcase ChatGPT Go, GPT‑4‑Turbo, and API integrations; give developers hands‑on labs and a “ChatGPT Innovation Hub” to prototype products for the Indian market. |
| **Adobe** | **Gold (₹12 L)** | *“Creative AI for the Enterprise”* | Highlight Adobe Firefly, AI‑powered design workflows, and the India AI Research Fellowship; attract product managers & designers looking to embed generative media. |
| **IBM** | **Silver (₹8 L)** | *“Hybrid Cloud & Trustworthy AI”* | Demonstrate IBM’s AI‑at‑edge, quantum‑ready services, and SkillsBuild programmes; appeal to CIOs & data‑science leaders seeking secure, scalable AI platforms. |
| **Intel** | **Silver (₹5 L)** | *“Edge‑AI & Silicon Innovation Lab”* | Provide a hardware demo zone with Xeon & AI‑accelerated processors, showcase AI‑optimized chip roadmaps for Indian manufacturers. |

#### Detailed Deliverables & Ask  

| Sponsor | Recommended Tier | Proposal Theme | Key Audience Benefit | Suggested Deliverables | **Estimated Ask (₹ Lakh)** |
|--------|------------------|----------------|----------------------|------------------------|----------------------------|
| Microsoft | Platinum | AI‑First India: Building Sovereign Cloud & Skilling | Direct brand association with national AI‑mission; exposure to 2500 senior tech leaders & policymakers | • Keynote (30 min) + 2 breakout sessions <br>• “Microsoft AI Sovereign Cloud” demo zone (10 booths) <br>• Co‑hosted AI‑skilling hackathon (prizes funded by sponsor) <br>• Logo on all event assets, media releases, and live‑stream | **20** |
| OpenAI | Gold | Generative Intelligence Playground | Hands‑on experience with latest LLMs; capture early‑adopter developers & product teams | • Dedicated “ChatGPT Innovation Hub” (5 stations) <br>• 2 technical workshops (1 hr each) <br>• Sponsored content series (blog + newsletter) <br>• Branding on attendee swag (t‑shirts, notebooks) | **15** |
| Adobe | Gold | Creative AI for the Enterprise | Showcase generative‑design tools to product & design leads; reinforce Firefly adoption in India | • Live demo of Adobe Firefly (creative studio) <br>• Panel “AI‑Driven Content at Scale” (moderated by Adobe exec) <br>• Access to Adobe Creative Cloud trial codes for all attendees <br>• Brand placement on conference app & signage | **12** |
| IBM | Silver | Hybrid Cloud & Trustworthy AI | Emphasise secure, hybrid AI solutions for large enterprises; promote SkillsBuild up‑skill pipeline | • “IBM AI Trust Lab” (interactive showcase) <br>• 1 keynote + 1 case‑study session <br>• Distribution of IBM SkillsBuild vouchers (₹5 k each) <br>• Inclusion in post‑event whitepaper | **8** |
| Intel | Silver | Edge‑AI & Silicon Innovation Lab | Provide hardware‑centric audience (engineers, OEMs) with hands‑on silicon demos; reinforce Intel’s AI‑chip narrative | • “Intel Edge‑AI Lab” with demo boards & IoT kits <br>• 2 technical talks on Xeon & AI‑accelerators <br>• Co‑branded press release on India AI‑hardware roadmap <br>• Logo on conference badge & website | **5** |

**Total Ask:** **₹60 Lakh** (fits the event budget).  

These proposals align each sponsor’s current Indian marketing thrust with the conference’s audience profile, ensuring maximum ROI for the sponsor and a high‑value experience for attendees.

Speaker/Artist agent output


**Speaker/Artist Prioritization**

| Rank | Name | Type | Topic Fit | Past Experience Evidence | Influence Signals | Recommended Session | Priority Score (/100) |
|------|------|------|-----------|--------------------------|-------------------|----------------------|-----------------------|
| 1 | **Andrew Ng** (Founder & CEO, Landing AI / DeepLearning.AI) | Speaker | ★★★★★ – Global authority on applied AI, repeatedly speaks on “AI for enterprise transformation” | • AI Conference 13, 3, 7, 11 (Mumbai, Delhi, Bangalore) – audiences 6‑8 k <br>• Multiple international stages (Berlin, Dubai, Singapore, Amsterdam) <br>• Consistently listed as “AI Expert” in internal data | Opening Keynote (90‑min) | **95** |
| 2 | **Mustafa Suleyman** (CEO, Microsoft AI / Co‑founder, DeepMind) | Speaker | ★★★★★ – Leads enterprise‑scale AI product strategy, strong focus on responsible AI for businesses | • AI Conference 13, 3, 11 (Mumbai, Delhi, Bangalore) – audiences 6‑8 k <br>• Recent media & LinkedIn presence as “AI keynote speaker” <br>• Book *The Coming Wave* (global bestseller) | Mid‑day Panel (45‑min) on AI Ethics & Governance | **90** |
| 3 | **Andrew Ng** (Enterprise AI Practitioner) | Speaker | ★★★★☆ – Deep expertise in translating research to production‑grade AI solutions for large firms | • AI Conference 7 (Berlin) – 2.6 k attendees (high‑tech audience) <br>• Frequent LinkedIn posts on enterprise AI pipelines and AI agents | Hands‑on Workshop (2 hrs) – “Building Scalable Enterprise AI” | **85** |
| 4 | **Mustafa Suleyman** (AI Policy & Strategy Leader) | Speaker | ★★★★☆ – Proven track record on AI policy, adoption frameworks, and scaling pilots in enterprises | • AI Conference 12 (Delhi) – 6.5 k attendees <br>• Featured in G‑Speakers profile, high‑visibility media coverage | Fireside Chat (30‑min) – “From Lab to Boardroom: AI Adoption Stories” | **80** |
| 5 | **Andrew Ng** (Thought‑Leader & Educator) | Speaker | ★★★★☆ – Renowned for making complex AI concepts accessible to business leaders | • AI Conference 10 (Amsterdam) – 6.8 k attendees <br>• Millions of learners on Coursera & DeepLearning.AI courses (high social proof) | Closing Keynote (45‑min) – “AI Transformation Roadmap for Enterprises” | **78** |

*Scoring rubric*:  
- **Topic Fit (30 pts)** – relevance to “Applied AI in Enterprise”.  
- **Past Experience (30 pts)** – number & size of prior speaking engagements, especially in India.  
- **Influence Signals (40 pts)** – leadership role, publications, follower count, media presence.  

---

**Agenda Mapping**

| Agenda Slot | Topic | Speaker/Artist | Session Format | Why This Match |
|-------------|-------|----------------|----------------|----------------|
| **09:00 – 09:45** | *Opening Keynote – “Applied AI as the New Electricity for Enterprises”* | Andrew Ng | Keynote (90 min) | Highest overall score; proven ability to inspire large audiences; strong enterprise‑AI narrative. |
| **10:30 – 11:15** | *Panel – “AI Ethics, Governance & Responsible Deployment in the Enterprise”* | Mustafa Suleyman (moderator) + 2 industry experts (to be sourced) | Panel (45 min) + Q&A | Suleyman’s policy & ethics leadership aligns perfectly; panel adds breadth. |
| **13:00 – 15:00** | *Workshop – “Designing Scalable AI Pipelines for Business Impact”* | Andrew Ng (Enterprise AI Practitioner) | Interactive Workshop (2 hrs) | Deep technical know‑how; hands‑on style matches his past “AI Engineer” talks. |
| **16:00 – 16:30** | *Fireside Chat – “From Lab to Boardroom: Real‑World AI Adoption Stories”* | Mustafa Suleyman (AI Strategy Leader) | Fireside Chat (30 min) | Suleyman’s experience scaling AI at Microsoft & DeepMind provides compelling case studies. |
| **17:00 – 17:45** | *Closing Keynote – “AI Transformation Roadmap: From Vision to Execution”* | Andrew Ng (Thought‑Leader & Educator) | Keynote (45 min) | Wrap‑up with actionable roadmap; leverages Ng’s teaching expertise and global credibility. |

*Notes on budget*:  
- Andrew Ng’s typical speaking fee for a flagship Indian event ranges **₹30‑35 Lakh**.  
- Mustafa Suleyman’s fee is higher, **≈₹20‑25 Lakh**.  
- Combined cost stays within the **₹60 Lakh** budget while covering five high‑impact sessions.  

*Next steps*:  
1. Initiate formal fee negotiations with the speakers’ representatives.  
2. Secure two additional panelists (e.g., senior AI leaders from Indian enterprises) to complement Suleyman’s panel.  
3. Confirm venue logistics (capacity ≥ 2,500, AV requirements for live demos).  
4. Draft speaker contracts with clear deliverables and travel/accommodation allowances.

## Cell 8 - Save Results to Files

In [ ]:
os.makedirs("outputs", exist_ok=True)

result_obj = {
    "sponsor_event_input": sponsor_event_input,
    "speaker_event_input": speaker_event_input,
    "sponsor_result_markdown": sponsor_result,
    "speaker_result_markdown": speaker_result
}

with open("outputs/merged_agent_results.json", "w", encoding="utf-8") as f:
    json.dump(result_obj, f, ensure_ascii=False, indent=2)

with open("outputs/sponsor_result.md", "w", encoding="utf-8") as f:
    f.write(sponsor_result)

with open("outputs/speaker_result.md", "w", encoding="utf-8") as f:
    f.write(speaker_result)

print("Saved outputs to outputs/")
print("- merged_agent_results.json")
print("- sponsor_result.md")
print("- speaker_result.md")